# 03 - Strategy comparison

Compare the baseline, rule-based, and Q-learning strategies on identical, seeded decks.
Each deck is played once by every strategy, so differences come from the strategy and not from lucky shuffles.

Requires matplotlib: `pip install -e ".[notebooks]"`. Run `blackjack train` first, or the Q-table is trained below.

In [ ]:
from pathlib import Path

from blackjack.ai.baseline import BaselineStrategy
from blackjack.ai.evaluation import compare_strategies
from blackjack.ai.q_learning import QLearningAgent
from blackjack.ai.rule_based import RuleBasedStrategy
from blackjack.ai.training import train

SEED = 42
GAMES = 1000

model = next((p for p in (Path("models/q_table.json"), Path("../models/q_table.json")) if p.exists()), None)
q_agent = QLearningAgent.load(model) if model else train(50_000, seed=SEED).agent
print("Q-table:", model or "trained in this notebook", f"({len(q_agent.q_table)} states)")

In [ ]:
results = compare_strategies(
    {"Baseline": BaselineStrategy(), "Rule-Based": RuleBasedStrategy(), "Q-Learning": q_agent},
    games=GAMES,
    seed=SEED,
)
print(f"{'Strategy':<12}{'Wins':>6}{'Losses':>8}{'Ties':>6}{'Win rate':>10}")
for r in results.values():
    print(f"{r.name:<12}{r.wins:>6}{r.losses:>8}{r.ties:>6}{r.win_rate:>10.1%}")

In [ ]:
import matplotlib.pyplot as plt

names = list(results)
rates = [100 * r.win_rate for r in results.values()]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(names, rates, color=["#9aa0a6", "#4c78a8", "#f58518"])
ax.bar_label(bars, fmt="%.1f%%")
ax.set_ylabel("Win rate (%)")
ax.set_ylim(0, max(rates) + 10)
ax.set_title(f"Win rate against the dealer ({GAMES:,} games each, seed {SEED})")
plt.tight_layout()
plt.show()

## Reading the result

Win rates over 1,000 games have a standard error of about 1.5 percentage points, so the gap between
rule-based and Q-learning is within noise; both clearly beat the baseline. Raise `GAMES` to tighten
the estimate, or change `SEED` to see how much it moves.